### 环境验证

In [53]:
import os
import time
import math
import pickle
import numpy as np
import torch

# 验证 CUDA 状态
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"当前 GPU 设备: {torch.cuda.get_device_name(0)}")
    # 4090 必须开启 TensorFloat-32 (TF32) 才能释放全力
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

CUDA 是否可用: False


In [54]:
config = {
    "dataset": "shakespeare_char",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "compile": True,           
    
    # hyperparameter
    "block_size": 256,           # 上下文最大长度 (Context length)
    "vocab_size": 65,            # 莎士比亚字符集的总字符数 (后面会动态读取更新)
    "n_layer": 6,                # Transformer 层数
    "n_head": 6,                 # 多头注意力的头数
    "n_embd": 384,               # 嵌入层维度 (每一个 token 的向量长度)
    "dropout": 0.2,
    
    # training
    "batch_size": 64,            # 每个 batch 的样本数
    "learning_rate": 1e-3,       # 初始学习率
    "max_iters": 2000,           # 总迭代步数
    "weight_decay": 1e-1,
    "beta1": 0.9,
    "beta2": 0.99,
    
    # eval
    "eval_interval": 200,        # 每隔多少步评估一次 loss
    "eval_iters": 50,            # 评估时估算多少个 batch
}

print("current device:", config["device"])

current device: cpu


### 1. 数据集下载和预处理

In [55]:
import urllib.request

data_dir = os.path.join('data', config['dataset'])
os.makedirs(data_dir, exist_ok=True)

input_file_path = os.path.join(data_dir, 'input.txt')

# download data
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(data_url, input_file_path)
    print("downloaded")


with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
print(f"data length: {len(data):,}")

# construct dictionary
chars = sorted(list(set(data)))
vocab_size = len(chars)
print(f"(Vocab Size): {vocab_size}")
print("table:", "".join(chars))

# update config
config['vocab_size'] = vocab_size

# 4. construct decoder and encoder between index and string
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # string to index
decode = lambda l: ''.join([itos[i] for i in l]) # index to string

# meta
meta = {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}
with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
    pickle.dump(meta, f)

n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

train_ids = np.array(encode(train_data), dtype=np.uint16)
val_ids = np.array(encode(val_data), dtype=np.uint16)

train_ids.tofile(os.path.join(data_dir, 'train.bin'))
val_ids.tofile(os.path.join(data_dir, 'val.bin'))
print("train.bin and val.bin ready")


data length: 1,115,394
(Vocab Size): 65
table: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
train.bin and val.bin ready


We construct a dataloader for batch division.

In [56]:
train_data_path = os.path.join(data_dir, 'train.bin')
val_data_path = os.path.join(data_dir, 'val.bin')

train_data_mem = np.memmap(train_data_path, dtype=np.uint16, mode='r')
val_data_mem = np.memmap(val_data_path, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data_mem if split == 'train' else val_data_mem
    # 切片生成batch size个数的一维tensor，其中每个数是训练开始的index，所以为了保证即使是很后面的index也能在block size下进行完整的训练，最后一个index应该是训练集token的总长度减去一个block size的大小
    index = torch.randint(len(data) - config['block_size'], (config['batch_size'],))
    
    # 得到多个不同的一维训练集然后叠batch size个
    x = torch.stack([torch.from_numpy((data[i:i+config['block_size']]).astype(np.int64)) for i in index])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+config['block_size']]).astype(np.int64)) for i in index])

    if config['device'] == 'cuda':
        x, y = x.pin_memory().cuda(non_blocking=True), y.pin_memory().cuda(non_blocking=True)
    else:
        x, y = x.to(config['device']), y.to(config['device'])
        
    return x, y

# --- testing ---
X, Y = get_batch('train')
print("X shape:", X.shape) # 预期: [batch_size, block_size] -> [64, 256]
print("Y shape:", Y.shape) # 预期: [64, 256]
print("\ntest:")
print(" X[0][:10]:", X[0][:10].tolist())
print(" Y[0][:10]:", Y[0][:10].tolist())

X shape: torch.Size([64, 256])
Y shape: torch.Size([64, 256])

test:
 X[0][:10]: [53, 56, 56, 53, 61, 1, 58, 46, 43, 52]
 Y[0][:10]: [56, 56, 53, 61, 1, 58, 46, 43, 52, 1]


### 2. 多头自注意力和Transformer

#### LayerNorm and MLP

In [57]:
import torch.nn as nn
from torch.nn import functional as F

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.MLP_stack = nn.Sequential(
            nn.Linear(config["n_embd"], 4 * config["n_embd"], bias = True),
            nn.GELU(),
            nn.Linear(4 * config["n_embd"],config["n_embd"], bias = True),
            nn.Dropout(config["dropout"])
        )
    
    def forward(self,x):
        return self.MLP_stack(x)

#### Causal Self-Attention

$$[B, T, C] \xrightarrow{\text{view + transpose}} [B, H, T, d] \xrightarrow{\text{Attention}} [B, H, T, d] \xrightarrow{\text{transpose + contiguous + view}} [B, T, C]$$
where B is batch size, T is sequence length, C is embedding dimension, H is number of heads, d is head size

In [58]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config["n_embd"] % config["n_head"] == 0
        self.attention = nn.Linear(config["n_embd"], 3 * config["n_embd"], bias = True)
        self.projection = nn.Linear(config['n_embd'], config['n_embd'], bias=True)

        self.attention_dropout = nn.Dropout(config['dropout'])
        self.residual_dropout = nn.Dropout(config['dropout'])
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']

    def forward(self, x):
        # obtain batch size, sequence length, embedding dimension as three integer
        B,T,C = x.size()

        # qkv shape: [B, T, 3 * C] -> 分裂成三个 [B, T, C]
        # shape: [B, T, n_head, head_size] -> 转置为 [B, n_head, T, head_size]
        q, k, v = self.attention(x).split(self.n_embd,dim = 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, 
            dropout_p=self.attention_dropout.p if self.training else 0.0, 
            is_causal=True # 声明为因果自注意力（自动应用下三角 Mask）
        )
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.residual_dropout(self.projection(y))
        return y




In [59]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config["n_embd"])
        self.attention = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config['n_embd'])
        self.mlp = MLP(config)

    def forward(self,x):
        #残差连接
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


In [60]:
# 测试代码
test_block = Block(config).to(config['device'])
# 模拟一个 Batch: batch_size=2, block_size=256, n_embd=384
dummy_input = torch.randn(2, config['block_size'], config['n_embd']).to(config['device'])

with torch.no_grad():
    dummy_output = test_block(dummy_input)

print("Input Shape:", dummy_input.shape)
print("Output Shape:", dummy_output.shape)
assert dummy_input.shape == dummy_output.shape, "wrong"
print("forward propagation success")

Input Shape: torch.Size([2, 256, 384])
Output Shape: torch.Size([2, 256, 384])
forward propagation success


### 3. GPT主类模型组装

In [64]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # Input embedding
        self.wte = nn.Embedding(config["vocab_size"],config["n_embd"])
        # Position encoding
        self.wpe = nn.Embedding(config['block_size'], config['n_embd'])
        # block stack
        self.h = nn.ModuleList([Block(config) for _ in range(config['n_layer'])])
        # final layernorm
        self.ln_f = nn.LayerNorm(config['n_embd'])
        # output head
        self.out = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)
        self.out.weight = self.wte.weight
        # 这是因为nn.Embedding和nn.Linear的初始幅度不同，nn.Linear的sd=0.03，而nn.Embedding的sd为1，所以我们用小方差初始化
        nn.init.normal_(self.wte.weight, mean=0.0, std=0.02)


    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        
        # 确保输入的序列长度没有超过我们允许的最大 block_size
        assert t <= self.config['block_size'], f" {t} > {self.config['block_size']}"
        
        # 生成位置序列 
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        
        # 计算两种嵌入并相加
        token_emb = self.wte(idx) # [b, t, n_embd]
        position_emb = self.wpe(pos) # [t, n_embd] 
        x = token_emb + position_emb  # [b, t, n_embd]
        
        for block in self.h:
            x = block(x)
            
        x = self.ln_f(x)
        
        if targets is not None:
            logits = self.out(x) # 形状: [b, t, vocab_size]
            # PyTorch 交叉熵要求输入是2维 [B*T, Vocab_Size]，标签是1维 [B*T]
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            # 推理/生成模式下，我们只关心最后一个时间步的输出，没必要算整个序列的 logits，省显存
            logits = self.out(x[:, [-1], :]) # 形状: [b, 1, vocab_size]
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config['block_size'] else idx[:, -self.config['block_size']:]
            
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # [b, vocab_size]
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1) # 形状: [b, 1]
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx
        

In [65]:
model = GPT(config).to(config['device'])

X, Y = get_batch('train')

logits, loss = model(X, Y)
print("Logits  (expect [64, 256, 65]):", logits.shape)
print("初始 Loss (expect 4.17):", loss.item())

# 4. generation test
context = torch.zeros((1, 1), dtype=torch.long, device=config['device']) # 传入 0 (通常是换行符)
generated_ids = model.generate(context, max_new_tokens=10)
print("测试生成 ID 序列:", generated_ids[0].tolist())

Logits  (expect [64, 256, 65]): torch.Size([64, 256, 65])
初始 Loss (expect 4.17): 4.24664306640625
测试生成 ID 序列: [0, 1, 50, 15, 36, 15, 50, 12, 23, 16, 3]


### 4. 训练循环

In [ ]:
import torch.optim as optim
from contextlib import nullcontext

# 我们不对所有的 LayerNorm 和 Bias 进行权重衰减，只对二维的矩阵乘法进行衰减
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}

# 分组：维度大于等于 2 的参数需要 decay（比如 Linear 的 weight），其余不需要（比如 bias, layernorm）
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': config['weight_decay']},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = optim.AdamW(optim_groups, lr=config['learning_rate'], betas=(config['beta1'], config['beta2']))

warmup_iters = 100 

def get_lr(it):
    if it < warmup_iters:
        return config['learning_rate'] * it / warmup_iters
    if it > config['max_iters']:
        return config['learning_rate'] * 0.1
    # 余弦退火
    decay_ratio = (it - warmup_iters) / (config['max_iters'] - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config['learning_rate'] * 0.1 + coeff * (config['learning_rate'] * 0.9)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() 
    for split in ['train', 'val']:
        losses = torch.zeros(config['eval_iters'])
        # 对于大型训练集随机抽取一定数量的batch求loss的平均值
        for k in range(config['eval_iters']):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() 
    return out


if config['compile']:
    print("compiling...")
    model = torch.compile(model)

ctx = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16) if config['device'] == 'cuda' else nullcontext()

print("start training...")
t0 = time.time()

for it in range(config['max_iters'] + 1):
    # 动态设定当前 step 的学习率
    lr = get_lr(it)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
        
    # 定期评估与文本生成展示
    if it % config['eval_interval'] == 0:
        losses = estimate_loss()
        t1 = time.time()
        print(f"Step {it}: 训练集 Loss {losses['train']:.4f}, 验证集 Loss {losses['val']:.4f}, 耗时: {t1-t0:.2f}s")
        t0 = time.time()
        
        # 让模型现场即兴创作一段（展示进化过程）
        print("-" * 40)
        # 拿一个换行符 ID (通常是 0) 作为 Prompt
        context_preview = torch.zeros((1, 1), dtype=torch.long, device=config['device'])
        # 如果启用了 torch.compile，推理时我们要调用原始未编译的模型内核（_orig_mod），避免触发多余的编译
        raw_model = model._orig_mod if config['compile'] else model
        raw_model.eval()
        generated_preview = raw_model.generate(context_preview, max_new_tokens=150)
        raw_model.train()
        print("模型现场生成效果:\n", decode(generated_preview[0].tolist()))
        print("-" * 40)

    # 捞出一个 Batch 的数据训练
    X, Y = get_batch('train')
    
    # 前向传播（包裹在混合精度中）
    with ctx:
        logits, loss = model(X, Y)
        
    # 反向传播与优化
    optimizer.zero_grad(set_to_none=True) # 比 zero_grad() 略快
    loss.backward()
    
    # 裁剪梯度，防止梯度爆炸 (大模型标配)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    optimizer.step()

正在编译模型 (torch.compile)... 第一次迭代会比较慢，请耐心等待


Process ForkProcess-5:
Process ForkProcess-2:
Process ForkProcess-3:
Process ForkProcess-8:
Process ForkProcess-7:
Process ForkProcess-4:
Process ForkProcess-6:
Process ForkProcess-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/process.

开始训练...


KeyboardInterrupt
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/connection.py", line 430, in _recv_bytes
    buf = self._recv(4)
          ^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/cam/lib/python3.11/multiprocessing/connection.py", line 395, in _recv
    chunk = read(handle, remaining)
            ^^^^^^^^^^^^^^^^^^^^^^^


KeyboardInterrupt: 